# 🎛️ Strata Studio

<a href="https://colab.research.google.com/github/PingPhantom/Colab/blob/main/nix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

จัดการโมเดล GGUF (10 ตัว + draft/MTP/N-gram) บน **llama.cpp** พร้อม **StrataRAG** และ UI 4 แท็บ
สำหรับ **Google Colab** และ **Kaggle Notebooks** (T4 ฟรีก็ใช้ได้)

**วิธีใช้:** รันเซลล์ 1 (ติดตั้งสภาพแวดล้อม) → เซลล์ 2 (เปิด UI) แล้วใช้ผ่านแท็บได้เลย

> โค้ดจริงอยู่ในแพ็กเกจ [`strata/`](./strata) — notebook เหลือแค่เซลล์ setup สั้น ๆ

> 🐨 **Kaggle:** เปิด *Settings → Internet on* ก่อน — โมเดลเก็บที่ `/kaggle/temp/strata` (ใหญ่กว่า `/kaggle/working` ที่จำกัด 20 GB) ตั้ง `STRATA_HOME=/kaggle/working` ถ้าอยากเก็บถาวร


In [ ]:
# @title 1) เตรียมโค้ด + ติดตั้งสภาพแวดล้อม (Nix + llama.cpp + uv) { display-mode: "form" }
# @markdown เปิด notebook เดี่ยว ๆ ใน Colab ได้เลย — เซลล์นี้จะ clone/อัปเดตโค้ดให้ถ้ายังไม่มีแพ็กเกจ `strata/`
REPO_SLUG = ""  # @param {type:"string"}
DEFAULT_SLUG = "PingPhantom/Colab"  # ใช้เมื่อเดาจาก URL ไม่ได้
REPO_BRANCH = ""  # @param {type:"string"}
AUTO_UPDATE = True  # @param {type:"boolean"}
# @markdown เว้น `REPO_SLUG`/`REPO_BRANCH` ว่าง = เดาให้อัตโนมัติ (URL ของ Colab → git ของ clone เดิม → PR ที่เปิดอยู่)
# @markdown `AUTO_UPDATE` = อัปเดตโค้ดให้ตรง GitHub ทุกครั้ง (Colab: `/content/Colab`, Kaggle: `/kaggle/working/Colab`) (การแก้โค้ดในเครื่องจะถูก reset ทิ้ง)

NOTEBOOK_REV = "v9-kaggle"  # ใช้ตรวจว่าโหลดเซลล์เวอร์ชันล่าสุดแล้ว

import importlib
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from urllib.parse import unquote

REQUIRED = ("strata/__init__.py", "pyproject.toml")  # ต้องมีครบถึงจะถือว่าโค้ดใช้ได้
GIT_ENV = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}


def _purge_modules() -> None:
    """ลบโมดูล `strata.*` ที่ค้างใน kernel — ไม่งั้นหลัง git fetch/reset จะยังรันไบต์โค้ดเก่าอยู่"""
    importlib.invalidate_caches()
    stale = [name for name in list(sys.modules) if name == "strata" or name.startswith("strata.")]
    for name in stale:
        del sys.modules[name]
    if stale:
        print(f"♻️ ล้างโมดูลเก่า {len(stale)} ตัว แล้วโหลดโค้ดจากดิสก์ใหม่")


def _ok(path) -> bool:
    return all((Path(path) / f).is_file() for f in REQUIRED)


def _git(args, cwd=None):
    return subprocess.run(["git", *(["-C", str(cwd)] if cwd else []), *args], check=True,
                          env=GIT_ENV, capture_output=True, text=True).stdout.strip()


def _pr_branches(slug: str) -> list:
    """ชื่อ branch ต้นทางของ PR ที่ยังเปิดอยู่ (ไว้ใช้ตอน branch หลักยังไม่มีแพ็กเกจ strata/)"""
    url = f"https://api.github.com/repos/{slug}/pulls?state=open&per_page=20"
    req = urllib.request.Request(url, headers={"Accept": "application/vnd.github+json",
                                               "User-Agent": "strata-colab"})
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:  # noqa: S310
            data = json.load(resp)
    except Exception as exc:  # noqa: BLE001
        print(f"⚠️ ถาม GitHub เรื่อง PR ไม่ได้ ({exc.__class__.__name__}) — ใช้ branch หลักเท่านั้น")
        return []
    return [p["head"]["ref"] for p in data if p.get("head", {}).get("ref")]


def _parse_github_url(url: str) -> tuple:
    """แยก (owner/repo, branch) จากลิงก์ Colab ที่เปิดจาก GitHub — branch มี "/" ได้"""
    if "/github/" not in (url or ""):
        return "", ""
    parts = unquote(url.split("/github/", 1)[1]).split("?")[0].split("#")[0].split("/")
    if len(parts) < 4 or parts[2] != "blob":
        return "", ""
    rest = parts[3:]
    if rest and rest[-1].endswith(".ipynb"):  # ตัดชื่อไฟล์ออก
        rest = rest[:-1]
    branch = "/".join(x for x in rest if x)
    return f"{parts[0]}/{parts[1]}", ("" if branch == "HEAD" else branch)


def _detect_source() -> tuple:
    """เดา (owner/repo, branch) จาก URL ของ Colab ก่อน ถ้าไม่ได้ค่อยดู git ของโฟลเดอร์ที่ clone ไว้"""
    url = ""
    try:  # เฉพาะบน Colab: อ่าน URL ของแท็บที่กำลังรันอยู่
        from google.colab import output  # type: ignore[import-not-found]

        url = output.eval_js("window.location.href") or ""
    except Exception:  # noqa: BLE001
        url = ""
    slug, branch = _parse_github_url(url)
    if slug:
        return slug, branch
    for base in (Path("/content/Colab"), Path.cwd() / "Colab", Path.cwd()):
        if (base / ".git").exists():
            try:
                ref = _git(["rev-parse", "--abbrev-ref", "HEAD"], cwd=base)
            except subprocess.CalledProcessError:
                continue
            if ref and ref != "HEAD":
                return "", ref
    return "", ""


def _refresh(target, refs) -> bool:
    """อัปเดต clone เดิมให้ตรงกับ ref (None = branch ปัจจุบัน) — เร็วและรักษา branch ที่ใช้อยู่"""
    for ref in refs:
        try:
            _git(["fetch", "--depth", "1", "origin", *([ref] if ref else [])], cwd=target)
            _git(["reset", "--hard", "FETCH_HEAD"], cwd=target)
        except subprocess.CalledProcessError:
            continue
        if _ok(target):
            print(f"🔄 อัปเดตโค้ดเดิมเป็น branch {ref or 'ล่าสุด'} เรียบร้อย")
            return True
    return False


def _ensure_repo(slug: str = "", branch: str = "", update=None) -> Path:
    """หาโฟลเดอร์ที่มีแพ็กเกจ `strata/` — ถ้าไม่มี/เก่าเกินไปให้ clone หรืออัปเดตให้ (Colab และเครื่องตัวเอง)"""
    here = Path.cwd()
    target = Path("/content/Colab") if Path("/content").is_dir() else here / "Colab"

    # 1) มี clone อยู่แล้ว -> fetch + reset ให้ตรง GitHub ก่อนเสมอ (กันโค้ดเก่าค้างข้ามรอบ)
    if (AUTO_UPDATE if update is None else update) and (target / ".git").exists():
        if _refresh(target, [branch] if branch else [None, *_pr_branches(slug)]):
            return target

    # 2) ไม่ได้อัปเดต (ปิด AUTO_UPDATE / ไม่ใช่ git) -> ใช้ของที่มีถ้าครบ
    for base in [target, here, *here.parents]:
        if _ok(base):
            return base

    # 3) ยังไม่มี -> clone ไล่ ref: ที่ระบุ > branch หลัก > branch ของ PR ที่เปิดอยู่
    refs = [branch] if branch else ["", *_pr_branches(slug)]
    print(f"⏳ ไม่พบแพ็กเกจ strata/ — กำลัง clone {slug} ...")
    for ref in refs:
        shutil.rmtree(target, ignore_errors=True)  # clone ค้าง/เก่า -> เริ่มใหม่
        cmd = ["git", "clone", "--depth", "1", *(["--branch", ref] if ref else [])]
        cmd += [f"https://github.com/{slug}.git", str(target)]
        subprocess.run(cmd, check=True, env=GIT_ENV)
        if _ok(target):
            print(f"✅ ใช้โค้ดจาก branch {ref or 'default'}")
            return target
        print(f"⚠️ branch {ref or 'default'} ยังไม่มีแพ็กเกจ strata/ — ลองตัวถัดไป")
    raise RuntimeError(
        f"clone {slug} แล้วไม่พบแพ็กเกจ strata/ สัก branch — ใส่ชื่อ branch ที่ต้องการในช่อง REPO_BRANCH "
        "(เช่น arena/e076c6f4-colab) หรือ merge PR เข้า branch หลักก่อน"
    )


print(f"🧩 strata notebook {NOTEBOOK_REV}")
_found_slug, _found_branch = _detect_source()
REPO_SLUG = REPO_SLUG or _found_slug or DEFAULT_SLUG
REPO_BRANCH = REPO_BRANCH or _found_branch
print(f"🌿 ใช้ {REPO_SLUG}" + (f" @ {REPO_BRANCH}" if REPO_BRANCH else " (branch หลัก/PR ที่เปิดอยู่)"))

REPO_ROOT = _ensure_repo(REPO_SLUG, REPO_BRANCH)
os.chdir(REPO_ROOT)  # flake.nix / pyproject.toml / .venv จะอยู่ที่รากรีโป
sys.path.insert(0, str(REPO_ROOT))
_purge_modules()  # ต้องล้างก่อน import: kernel อาจถือโค้ดเวอร์ชันก่อน git reset อยู่
print(f"📂 ใช้โค้ดจาก: {REPO_ROOT} (commit {_git(['rev-parse', '--short', 'HEAD'], cwd=REPO_ROOT)})")

from strata.bootstrap import ensure_environment

t0 = time.perf_counter()
env = ensure_environment()
print(f"✅ ใช้เวลาเตรียม env {time.perf_counter() - t0:.1f}s • nixpkgs {env['nixpkgs'][:7]}")


In [ ]:
# @title 2) เปิด Studio UI (พอร์ต 18080) { display-mode: "form" }
import os
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents, Path("/content/Colab"), Path("/kaggle/working/Colab")]:
    if (_base / "strata" / "__init__.py").is_file():
        sys.path.insert(0, str(_base))
        os.chdir(_base)
        break
else:
    raise RuntimeError("ยังไม่พบแพ็กเกจ strata/ — รันเซลล์ 1 ก่อนครับ")

from strata.notebook import diagnose, launch, register_magics

register_magics()  # เปิดใช้ %%ruff ในเซลล์ใหม่ (ทางเลือก)

info = diagnose()  # พิมพ์สรุป: แพลตฟอร์ม, path, เวอร์ชันวิดเจ็ต, ดิสก์
ui = launch()      # ถ้าวิดเจ็ตแสดงไม่ได้ จะพิมพ์ dashboard แบบข้อความแทนให้อัตโนมัติ


## ใช้งานผ่านโค้ด (ไม่ต้องเปิด UI)

```python
from strata.download import install_model, remove_model
from strata.server import start_strata_server, stop_strata_server, get_openai_client

install_model("qwen35-9b-defiant-fable", auto_start_server=True)   # โหลด + สตาร์ท
client = get_openai_client()                                       # OpenAI-compatible ที่ :18080
print(client.chat.completions.create(
    model="qwen35-9b-defiant-fable",
    messages=[{"role": "user", "content": "สวัสดี"}],
).choices[0].message.content)
```

เรียก API จาก notebook/แอปอื่นได้ที่ `http://127.0.0.1:18080/v1` (OpenAI SDK, `curl`, LangChain ฯลฯ)

## RAG

```python
from strata.rag import GLOBAL_RAG

GLOBAL_RAG.add_files("/content/doc.pdf", chunk_size=500)
GLOBAL_RAG.save()          # /content/rag_store.npz (ชี้ไป Drive ได้)
print(GLOBAL_RAG.search("คำถาม", top_k=3))
```

## ตั้งค่าผ่าน environment variable

| ตัวแปร | ค่าเริ่มต้น | ใช้ทำอะไร |
| --- | --- | --- |
| `STRATA_PORT` | `18080` | พอร์ตของ API server/UI |
| `STRATA_HOME` | `/content` | โฟลเดอร์เก็บ log/คลัง RAG |
| `STRATA_MODEL_DIR` | `$STRATA_HOME/models` | ที่เก็บโมเดล GGUF |
| `STRATA_RAG_STORE` | `$STRATA_HOME/rag_store.npz` | ไฟล์คลัง RAG |
| `LLAMA_CPP_TAG` | release ล่าสุด | ปักหมุดเวอร์ชัน llama.cpp |
| `NIX_INSTALL_SHA256` | – | บังคับ checksum ของสคริปต์ติดตั้ง Nix |

รายละเอียดทั้งหมด (ข้อกำหนดเครื่อง, เทสต์, โครงสร้างโค้ด) อยู่ใน [`README.md`](./README.md)
